# BigQuery Data Profile & Data Quality 자동화 파이프라인

이 노트북은 Google Cloud의 Dataplex DataScan API를 사용하여 `thelook_ecommerce` 데이터셋에 속한 주요 테이블들의 **데이터 프로파일(Data Profile)** 및 **데이터 품질(Data Quality)** 스캔을 자동 생성, 실행하고 결과를 모니터링합니다.

In [ ]:
# 라이브러리 임포트
import json
import time
import subprocess
import urllib.request
import urllib.error
import ssl
from google.cloud import bigquery

# 1. GCP 프로젝트 및 리전 정보 정의
# gcloud CLI 명령어를 쉘에서 직접 호출하여, 현재 로그인된(active) 구글 클라우드 프로젝트 ID를 자동으로 가져옵니다.
project_proc = subprocess.run(
    ["gcloud", "config", "get-value", "project"],
    capture_output=True, text=True, check=True
)
# 경고 문구(WARNING) 라인을 제외하고 실제 프로젝트 ID가 담긴 마지막 라인을 파싱합니다.
project_lines = [line.strip() for line in project_proc.stdout.split("\n") if line.strip() and not line.strip().startswith("WARNING:")]
PROJECT_ID = project_lines[-1]

DATASET_ID = "thelook_ecommerce" # 분석 대상 빅쿼리 데이터셋명
LOCATION = "asia-northeast3"     # Knowledge Catalog DataScan이 실행 및 생성될 GCP 리전 위치 (서울 리전)

# 2. Knowledge Catalog REST API 호출용 OAuth2 Access Token 발급
# gcloud CLI를 사용하여 현재 활성화된 사용자/서비스 계정의 인증 토큰을 임시 발급받아 REST API 헤더에 넣을 준비를 합니다.
token_proc = subprocess.run(
    ["gcloud", "auth", "print-access-token"],
    capture_output=True, text=True, check=True
)
token_lines = [line.strip() for line in token_proc.stdout.split("\n") if line.strip() and not line.strip().startswith("WARNING:")]
ACCESS_TOKEN = token_lines[-1]

# 인증서 확인 우회를 위한 SSL 컨텍스트 설정 (테스트 및 실습 목적)
ssl_context = ssl._create_unverified_context()

# BigQuery API 호출용 공식 Python SDK 클라이언트 초기화
bq_client = bigquery.Client(project=PROJECT_ID)

print(f"현재 인식된 Project ID: {PROJECT_ID}, 대상 Dataset: {DATASET_ID}")
print("GCP 환경 및 API 인증 설정 완료!")

In [ ]:
# [공통 유틸리티] Knowledge Catalog REST API 전송 함수
# Knowledge Catalog(Dataplex)의 일부 고급/프리뷰 기능은 공식 파이썬 SDK에서 완벽하게 지원되지 않으므로, 표준 REST API 통신을 통해 제어합니다.
def make_rest_request(url, method="GET", body_dict=None):
    req = urllib.request.Request(url, method=method)
    # 1단계에서 발급받은 OAuth2 임시 토큰을 Authorization Bearer 헤더에 주입하여 API 인증을 통과합니다.
    req.add_header("Authorization", f"Bearer {ACCESS_TOKEN}")
    req.add_header("Content-Type", "application/json")
    
    data = None
    if body_dict:
        # 파이썬 딕셔너리 페이로드를 JSON 문자열로 변환하여 POST 전송 준비를 합니다.
        data = json.dumps(body_dict).encode("utf-8")
        
    try:
        with urllib.request.urlopen(req, data=data, context=ssl_context) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        # API 호출 중 에러가 발생한 경우 상세 에러 응답 바디를 파싱하여 예외를 발생시킵니다.
        err_msg = e.read().decode("utf-8")
        raise Exception(f"HTTP Error {e.code} - {err_msg}")

# [공통 유틸리티] DataScan Job 비동기 기동 및 대기 함수
def run_datascan_and_wait(scan_id):
    """
    정의된 DataScan(프로파일러 또는 퀄리티 규칙)의 실행 Job을 구동하고 완료될 때까지 대기(폴링)합니다.
    """
    # 1. 특정 DataScan을 수동으로 일회성 실행(:run)하기 위한 POST API 요청 URL 정의
    run_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}:run"
    print(f"  -> DataScan 실행 요청 중 (Scan ID: {scan_id})...")
    run_res = make_rest_request(run_url, method="POST")
    
    # 2. 기동된 비동기 Job의 이름(프로젝트, 리전, 스캔ID 및 고유 Job ID 조합) 파싱
    job_name = run_res["job"]["name"]
    job_id = run_res["job"]["uid"]
    print(f"  -> 실행 Job ID: {job_id} (완료될 때까지 상태를 폴링하기 시작합니다...)")
    
    # 3. 비동기 백엔드 작업 진행률 모니터링을 위한 상태 폴링 루프
    job_url = f"https://dataplex.googleapis.com/v1/{job_name}"
    while True:
        # 생성된 Job의 최신 진행 상태를 GET 조회
        job = make_rest_request(job_url, method="GET")
        state = job.get("state")
        print(f"     [폴링] 현재 작업 상태: {state}")
        
        # 완료 상태가 'SUCCEEDED'일 때 결과 데이터를 리턴
        if state == "SUCCEEDED":
            print("  -> 스캔 완료!")
            return job
        # 실패 또는 취소와 같은 비정상 종료 시 예외를 발생
        elif state in ["FAILED", "CANCELLED"]:
            raise Exception(f"DataScan Job 실행 오류 발생 (상태: {state})")
            
        # 10초 대기 후 재조회 (Google Cloud API의 Rate Limit을 준수하기 위한 간격)
        time.sleep(10)

# [공통 유틸리티] DataScan Job 비동기 기동 함수 (대기 없음)
def trigger_datascan(scan_id):
    """
    정의된 DataScan의 실행 Job을 구동하고 완료될 때까지 대기하지 않고 Job 이름을 반환합니다.
    """
    run_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}:run"
    run_res = make_rest_request(run_url, method="POST")
    return run_res["job"]["name"]

## 1. 데이터 프로파일 (Data Profile) 자동화

데이터 프로파일러는 컬럼별 데이터 타입 분석, Null 비율, 고유값 비율(Cardinality), 최솟값/최댓값/평균값 등 통계치를 자동으로 수집합니다.

> [!TIP]
> **비용 최적화 (Sampling)**: 모든 데이터를 스캔하는 것은 비용이 많이 들 수 있습니다. `samplingPercent` 옵션을 사용해 데이터의 일부(예: 10% 또는 1%)만 샘플링하여 분석함으로써 스캔 비용을 획기적으로 낮출 수 있습니다. 본 노트북에서는 기본값으로 **10% (`10.0`)**를 사용합니다.

In [ ]:
# [Data Profile] 데이터 프로파일 스캔 조회 및 생성 함수
def get_or_create_data_profile_scan(table_id, sampling_percent=10.0):
    """
    대상 빅쿼리 테이블에 대한 DATA_PROFILE Knowledge Catalog DataScan 리소스를 조회하고, 없으면 새로 생성합니다.
    분석 결과는 지정된 BigQuery 데이터셋 내의 'dataplex_profile_results' 테이블로 자동 내보내기(Export)되도록 구성합니다.
    """
    # Knowledge Catalog DataScan 리소스 ID는 영문 소문자, 숫자, 대시(-)만 허용하므로 밑줄(_)을 변환합니다.
    scan_id = f"dp-thelook-{table_id}".lower().replace("_", "-")
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    
    try:
        # 기존에 등록된 스캔이 있는지 검사
        scan = make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Data Profile Scan 발견: {scan_id}")
        return scan_id
    except Exception as e:
        # 404 (Not Found) 에러 발생 시, 신규 스캔 생성 로직 진입
        if "404" in str(e):
            print(f"  -> 새로운 Data Profile Scan 생성 중: {scan_id} (분석 대상 비율: {sampling_percent}%)...")
            create_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}"
            
            # 스캔 완료 결과를 주기적으로 축적할 빅쿼리 결과 테이블 주소 정의
            export_table = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/dataplex_profile_results"
            
            # Knowledge Catalog DataScan API 요청 바디 구성
            # postScanActions는 dataProfileSpec 하위에 위치해야 스펙에 맞습니다.
            body = {
                "data": {
                    # 분석 대상 물리 리소스 (BigQuery 테이블) 지정
                    "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
                },
                "executionSpec": {
                    # 매번 사용자가 필요할 때 수동으로 구동하기 위해 온디맨드(onDemand) 트리거 설정
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_PROFILE", # 데이터 프로파일 타입 지정
                "dataProfileSpec": {
                    "samplingPercent": sampling_percent, # 연산 및 데이터 처리 비용 경감을 위해 샘플링 비율 설정
                    "postScanActions": {
                        "bigqueryExport": {
                            # 분석 완료된 프로파일 보고서 로우 데이터를 빅쿼리 테이블로 내보내는 동작 정의
                            "resultsTable": export_table
                        }
                    },
                    "catalogPublishingEnabled": True
                }
            }
            # LRO (Long Running Operation) 요청 생성
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            
            # 리소스 생성이 백엔드에서 최종 완료될 때까지 Operation 상태 폴링 대기
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    if "error" in op_status:
                        raise Exception(f"Data Profile Scan 생성 실패: {op_status['error']}")
                    break
                time.sleep(2)
            print(f"  -> Data Profile Scan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

# [Data Profile] 프로파일 요약 결과 출력 함수
def print_profile_summary(job_data):
    """
    실행 완료된 Knowledge Catalog Job 결과 페이로드에서 프로파일 정보(통계값)를 추출하여 텍스트 포맷으로 가독성 있게 출력합니다.
    """
    result = job_data.get("dataProfileResult", {})
    print(f"\n[Data Profile 요약] (총 로우 수: {result.get('rowCount', 'N/A')})")
    
    # 각 컬럼(필드)별 상세 분석값 파싱
    fields = result.get("profile", {}).get("fields", [])
    for field in fields:
        name = field.get("name")
        f_type = field.get("type")
        profile = field.get("profile", {})
        
        # Null 값의 백분율 비율 계산
        null_ratio = profile.get("nullRatio", 0) * 100
        # 고유값(Distinct) 비율 계산 (카디널리티 분석)
        distinct_ratio = profile.get("distinctRatio", 0) * 100
        
        print(f" - 컬럼: {name} ({f_type})")
        print(f"   * Null 비율: {null_ratio:.2f}%")
        print(f"   * 고유값 비율: {distinct_ratio:.2f}%")
        
        # 데이터의 숫자 유형에 따른 최대값, 최소값, 평균값 추가 요약 출력
        if "integerProfile" in profile:
            p = profile["integerProfile"]
            print(f"   * 통계치: Min={p.get('min')}, Max={p.get('max')}, Avg={p.get('average')}")
        elif "doubleProfile" in profile:
            p = profile["doubleProfile"]
            print(f"   * 통계치: Min={p.get('min')}, Max={p.get('max')}, Avg={p.get('average')}")

## 2. 데이터 품질 (Data Quality) 자동화

데이터 품질 스캔은 스키마에 정의된 제약조건(Null 불가, 고유키, 범위 지정 등)에 부합하는지 규칙(Rules)을 만들어 검증합니다.

In [ ]:
# [Data Quality] 빅쿼리 스키마 분석 기반 동적 품질 규칙(Rule) 생성기
def generate_default_quality_rules(table_id):
    """
    빅쿼리 테이블의 실제 스키마 정보를 실시간으로 읽어와 논리적 기본 데이터 품질 규칙(Rule)을 동적으로 구성합니다.
    이를 통해 하드코딩 없이 테이블 스펙 변화에 유연하게 대응합니다.
    """
    table_ref = bq_client.dataset(DATASET_ID).table(table_id)
    table = bq_client.get_table(table_ref)
    
    rules = []
    
    # 테이블 내의 컬럼 정보 리스트를 순회하며 적합한 규칙 정의
    for field in table.schema:
        # 1. COMPLETENESS (완전성) 차원: REQUIRED(필수값) 컬럼은 NULL 값을 허용하지 않는 규칙 추가
        if field.mode == "REQUIRED":
            rules.append({
                "column": field.name,
                "dimension": "COMPLETENESS",
                "nonNullExpectation": {}, # NULL 불가 조건 설정
                "description": f"{field.name} 컬럼은 필수값이므로 NULL을 허용하지 않습니다."
            })
            
        # 2. UNIQUENESS (유일성) 차원: 컬럼명이 고유 키인 'id'일 경우 중복값을 허용하지 않는 규칙 추가
        if field.name == "id":
            rules.append({
                "column": field.name,
                "dimension": "UNIQUENESS",
                "uniquenessExpectation": {}, # 중복 불가 조건 설정
                "description": f"{field.name} 컬럼은 고유 레코드 식별자이므로 중복을 허용하지 않습니다."
            })
            
        # 3. VALIDITY (유효성) 차원: 위도 및 경도 지리 정보 컬럼의 값 범위(-90~90, -180~180) 한계 검증 규칙 추가
        if field.name == "latitude" and field.field_type in ("FLOAT", "INTEGER"):
            rules.append({
                "column": field.name,
                "dimension": "VALIDITY",
                "rangeExpectation": {
                    "minValue": "-90",
                    "maxValue": "90"
                },
                "description": f"위도 좌표는 유효 범위(-90 ~ 90) 내에 있어야 합니다."
            })
        elif field.name == "longitude" and field.field_type in ("FLOAT", "INTEGER"):
            rules.append({
                "column": field.name,
                "dimension": "VALIDITY",
                "rangeExpectation": {
                    "minValue": "-180",
                    "maxValue": "180"
                },
                "description": f"경도 좌표는 유효 범위(-180 ~ 180) 내에 있어야 합니다."
            })
            
    return rules

# [Data Quality] 데이터 품질 스캔 조회 및 생성 함수
def get_or_create_data_quality_scan(table_id):
    """
    대상 테이블에 대한 DATA_QUALITY Knowledge Catalog DataScan 리소스를 조회하고, 없으면 생성합니다.
    검증 결과는 지정된 BigQuery 데이터셋 내의 'dataplex_quality_results' 테이블로 내보냅니다.
    """
    scan_id = f"dq-thelook-{table_id}".lower().replace("_", "-")
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    
    try:
        scan = make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Data Quality Scan 발견: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e):
            print(f"  -> 새로운 Data Quality Scan 생성 중: {scan_id}...")
            create_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}"
            
            # 동적으로 데이터 품질 검사 규칙 리스트 확보
            rules = generate_default_quality_rules(table_id)
            if not rules:
                # 테이블에 마땅한 특수 규칙이 검출되지 않는 경우, 임시로 첫 번째 컬럼에 기본 NULL 체크 규칙을 심어 에러를 방지합니다.
                fields_list = [f.name for f in bq_client.get_table(bq_client.dataset(DATASET_ID).table(table_id)).schema]
                check_col = "id" if "id" in fields_list else bq_client.get_table(bq_client.dataset(DATASET_ID).table(table_id)).schema[0].name
                rules = [{
                    "column": check_col,
                    "dimension": "COMPLETENESS",
                    "nonNullExpectation": {},
                    "description": "임시 기본 NULL 방지 체크"
                }]
                
            export_table = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/dataplex_quality_results"
            
            # DataScan 유형을 'DATA_QUALITY'로 정의하고 동적 생성된 규칙 목록 주입
            # postScanActions는 dataQualitySpec 하위에 위치해야 스펙에 맞습니다.
            body = {
                "data": {
                    "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_QUALITY",
                "dataQualitySpec": {
                    "rules": rules,
                    "postScanActions": {
                        "bigqueryExport": {
                            # 분석 완료된 프로파일 보고서 로우 데이터를 빅쿼리 테이블로 내보내는 동작 정의
                            "resultsTable": export_table
                        }
                    },
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    if "error" in op_status:
                        raise Exception(f"Data Quality Scan 생성 실패: {op_status['error']}")
                    break
                time.sleep(2)
            print(f"  -> Data Quality Scan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

# [Data Quality] 품질 요약 결과 출력 함수
def print_quality_summary(job_data):
    """
    품질 검증 Knowledge Catalog Job 결과를 분석하여 전체 결과와 각 차원/규칙별 세부 성공 통계를 예쁘게 포매팅하여 출력합니다.
    """
    result = job_data.get("dataQualityResult", {})
    passed = result.get("passed", False)
    print(f"\n[Data Quality 결과]: {'통과 (PASSED)' if passed else '실패 (FAILED) ❌'}")
    
    # 1. 고수준 차원(Completeness, Uniqueness, Validity 등) 단위 성공 여부 출력
    dimensions = result.get("dimensions", [])
    print(" - 차원별 통과 여부:")
    for dim in dimensions:
        print(f"   * {dim.get('dimension')}: {'성공 ✅' if dim.get('passed') else '실패 ❌'}")
        
    # 2. 로우 레벨 세부 규칙별 검증 통계 파싱 및 출력
    rules = result.get("rules", [])
    print(" - 규칙별 세부 검증 통계:")
    for r in rules:
        r_info = r.get("rule", {})
        rule_desc = r_info.get("description", "")
        col = r_info.get("column", "")
        passed_rule = r.get("passed", False)
        
        evaluated = r.get("evaluatedCount", 0) # 평가가 수행된 전체 레코드 수
        passed_cnt = r.get("passedCount", 0)   # 검증을 정상 통과한 성공 레코드 수
        null_cnt = r.get("nullCount", 0)       # 평가 중 Null이 검출된 레코드 수
        
        print(f"   * [{col}] {rule_desc} -> {'통과' if passed_rule else '실패'} (평가수={evaluated}, 통과수={passed_cnt}, NULL수={null_cnt})")

## 3. 테스트 실행 (단일 테이블)

예시 테이블(`distribution_centers`)을 통해 데이터 프로파일 및 품질 스캔을 구동해봅니다.

In [ ]:
# [검증 테스트] 단일 테이블에 대한 스캔 기능 테스트 구동
# 상대적으로 크기가 작아 연산이 빠른 물류센터('distribution_centers') 테이블을 샘플로 선택하여 파이프라인 작동 여부를 테스트합니다.

TEST_TABLE = "distribution_centers"

print(f"=== [{TEST_TABLE}] 데이터 프로파일 및 품질 검사 시뮬레이션 시작 ===")

# 1. 데이터 프로파일 생성 및 비동기 실행 (데이터 비용 절감을 위해 10% 샘플링 기동)
dp_scan_id = get_or_create_data_profile_scan(TEST_TABLE, sampling_percent=10.0)
dp_job = run_datascan_and_wait(dp_scan_id) # Job 기동 후 완료 상태(SUCCEEDED)까지 폴링 대기
print_profile_summary(dp_job)              # 완료 후 요약 리포트 파싱 출력

# 2. 데이터 품질 검사 생성 및 비동기 실행 (Knowledge Catalog)
dq_scan_id = get_or_create_data_quality_scan(TEST_TABLE)
dq_job = run_datascan_and_wait(dq_scan_id) # Job 기동 후 완료 상태(SUCCEEDED)까지 폴링 대기
print_quality_summary(dq_job)              # 완료 후 요약 리포트 파싱 출력

## 4. 전체 테이블 데이터 분석 일괄 적용

모든 테이블(`users`, `orders`, `order_items`, `products`, `events`, `inventory_items`)에 일괄 적용하는 부분입니다. (각 스캔에 약 1~2분 소요)

In [ ]:
# [메인 실행 파이프라인] 전체 테이블 대상 일괄 메타데이터 구축 실행
# 시뮬레이션 검증이 완료되었으므로, 전체 테이블들을 비동기(Async) 병렬 기동하고 요약 모니터링을 진행합니다.

ALL_TABLES = [
    "users",
    "orders",
    "order_items",
    "products",
    "events",
    "inventory_items"
]

print("=== [전체 테이블 대상 데이터 프로파일 / 품질 스캔 자동화 비동기 일괄 기동] ===")

active_jobs = {} # {job_name: (table_id, scan_type)}

for t_id in ALL_TABLES:
    print(f"  -> '{t_id}' 테이블 스캔 기동 중...")
    try:
        # A. 프로파일 스캔 조회/생성 및 실행 (10% 샘플링 적용)
        dp_scan = get_or_create_data_profile_scan(t_id, sampling_percent=10.0)
        dp_job_name = trigger_datascan(dp_scan)
        active_jobs[dp_job_name] = (t_id, "Profile")
        
        # B. 품질 검증 스캔 조회/생성 및 실행
        dq_scan = get_or_create_data_quality_scan(t_id)
        dq_job_name = trigger_datascan(dq_scan)
        active_jobs[dq_job_name] = (t_id, "Quality")
        
    except Exception as e:
        print(f"  [오류 발생] '{t_id}' 테이블 기동 실패: {e}")

total_triggered = len(active_jobs)
print(f"\n총 {total_triggered}개의 데이터 스캔 작업이 동시에 기동되었습니다. 진행 상황 모니터링을 시작합니다.")
print("(개별 작업 완료 시 알림이 출력되며, 30초 주기로 전체 현황을 요약 출력합니다.)\n")

# 간결한 진행 상태 모니터링 루프
while active_jobs:
    status_summary = {"PENDING": 0, "RUNNING": 0, "SUCCEEDED": 0, "FAILED": 0, "CANCELLED": 0}
    
    # 작업 상태 확인 및 처리 완료 대상 식별
    for job_name in list(active_jobs.keys()):
        job_url = f"https://dataplex.googleapis.com/v1/{job_name}"
        try:
            job = make_rest_request(job_url, method="GET")
            state = job.get("state", "UNKNOWN")
            
            status_summary[state] = status_summary.get(state, 0) + 1
            
            if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
                t_id, s_type = active_jobs[job_name]
                print(f"  [완료] '{t_id}' {s_type} 작업 종료 (결과: {state})")
                del active_jobs[job_name]
        except Exception as e:
            # 네트워크 순간 오류 등으로 인한 일시적 에러는 로깅 후 무시하고 다음 루프에 재시도
            pass
            
    if active_jobs:
        running_cnt = status_summary.get("RUNNING", 0) + status_summary.get("PENDING", 0)
        succeeded_cnt = total_triggered - len(active_jobs) - status_summary.get("FAILED", 0) - status_summary.get("CANCELLED", 0)
        
        print(f"[진행 상태] 진행 중: {running_cnt}개 | 완료: {succeeded_cnt}개 | 실패: {status_summary.get('FAILED', 0)}개", end="\r")
        time.sleep(30)

print("\n\n=== [모든 테이블 데이터 스캔 자동화 파이프라인 일괄 실행 완료] ===")